# Build APK - Bể Cá 3D (Three.js + Capacitor)
Dựa trên quy trình `Midnight Expressway - Build APK.ipynb`:
- Nén model sang `assets/models-optimized/` và **loại bỏ `assets/models/`** khỏi gói APK.
- Tự động tạo icon Android từ `assets/icon.png`.
- Build `app-debug.apk`, lưu lên Google Drive và tải về máy.

In [ ]:
%%bash
set -euo pipefail
# Lấy bản game trên GitHub và cài bộ công cụ build.
cd /content
if [ ! -d Be-ca-3D ]; then
  GIT_TERMINAL_PROMPT=0 git clone --depth 1 --branch main https://github.com/spdatdd/vibe-code-tank-fish-3d.git Be-ca-3D
fi
test -f Be-ca-3D/index.html
sudo apt-get update -qq
sudo apt-get install -y -qq openjdk-21-jdk-headless
mkdir -p /content/node /content/android-sdk/cmdline-tools/latest
curl -fsSL https://nodejs.org/dist/v22.21.1/node-v22.21.1-linux-x64.tar.xz | tar -xJ --strip-components=1 -C /content/node
tmp_dir=$(mktemp -d)
curl -fsSL https://dl.google.com/android/repository/commandlinetools-linux-15859902_latest.zip -o "$tmp_dir/tools.zip"
unzip -q "$tmp_dir/tools.zip" -d "$tmp_dir"
cp -a "$tmp_dir/cmdline-tools/." /content/android-sdk/cmdline-tools/latest/
printf '\nBUILD TOOLS READY\n'

In [ ]:
%%bash
set -euo pipefail
# Dùng Temurin 21 để tránh cảnh báo cgroup của JDK mặc định khi Gradle gọi jlink.
export JAVA_HOME=/content/temurin21
if [ ! -x "$JAVA_HOME/bin/jlink" ]; then
  mkdir -p "$JAVA_HOME"
  curl -fLsS --retry 2 'https://github.com/adoptium/temurin21-binaries/releases/download/jdk-21.0.8%2B9/OpenJDK21U-jdk_x64_linux_hotspot_21.0.8_9.tar.gz' | tar -xz --strip-components=1 -C "$JAVA_HOME"
fi
"$JAVA_HOME/bin/jlink" --version
export ANDROID_HOME=/content/android-sdk
export PATH="/content/node/bin:$JAVA_HOME/bin:$PATH"
# Chấp nhận giấy phép Android SDK rồi cài các gói cần cho build.
sdkmanager="$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager"
if [ ! -d "$ANDROID_HOME/licenses" ]; then
  (set +o pipefail; yes | "$sdkmanager" --sdk_root="$ANDROID_HOME" --licenses)
fi
"$sdkmanager" --sdk_root="$ANDROID_HOME" 'platforms;android-36' 'build-tools;36.0.0' 'platform-tools'

# Tối ưu model (nếu có assets/models), đóng gói web assets vào thư mục www/.
cd /content/Be-ca-3D
if [ -d assets/models ]; then
  npm ci || npm install
  npm run optimize:models
fi
rm -rf www && mkdir -p www
cp index.html www/
cp -r lib assets www/
rm -rf www/assets/models
rm -f www/assets/icon.png
ls -lh www/ www/assets/

# Nhúng www vào ứng dụng Android, tạo icon và build APK debug.
npm install @capacitor/core@8 @capacitor/android@8
npm install -D @capacitor/cli@8 @capacitor/assets
if ! compgen -G 'capacitor.config.*' >/dev/null; then
  npx cap init 'Be Ca 3D' com.spdatdd.beca3d --web-dir www
fi
if [ ! -d android ]; then npx cap add android; fi
if [ -f assets/icon.png ]; then
  mkdir -p resources
  cp assets/icon.png resources/icon.png
  npx @capacitor/assets generate --android --assetPath resources
fi
npx cap sync android
cd android
bash gradlew assembleDebug --no-daemon
ls -lh app/build/outputs/apk/debug/app-debug.apk

In [ ]:
from pathlib import Path
from shutil import copy2
from google.colab import drive

# Lưu đúng tệp APK đã build vào thư mục gốc của Drive.
source = Path('/content/Be-ca-3D/android/app/build/outputs/apk/debug/app-debug.apk')
assert source.is_file(), 'Không tìm thấy APK trong runtime Colab.'
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/Be-Ca-3D-debug.apk')
if destination.exists():
    from uuid import uuid4
    destination = destination.with_stem(f'{destination.stem}-{uuid4().hex[:8]}')
copy2(source, destination)
assert destination.stat().st_size == source.stat().st_size
print(f'ĐÃ LƯU VÀO DRIVE: {destination.name} ({destination.stat().st_size:,} byte)')

In [ ]:
from google.colab import files

# Tải APK debug về máy sau khi Gradle build thành công.
files.download('/content/Be-ca-3D/android/app/build/outputs/apk/debug/app-debug.apk')